### Camada Bronze

A camada Bronze tem como objetivo armazenar os dados em seu estado original, preservando as características da fonte para posterior análise e tratamento nas etapas subsequentes do pipeline.

In [0]:
%python
# ==============================
# IMPORTAÇÕES
# ==============================

import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, sum, when
from pyspark.sql import functions as F
#import seaborn as sns
from matplotlib.ticker import FuncFormatter

In [0]:
%python
# ==============================
# FUNÇÕES ÚTEIS
# ==============================

def importar_dataset():
   urlDados = 'https://raw.githubusercontent.com/marciomalrj/MVPED/main/Vendas.CSV'
   df = pd.read_csv(urlDados, sep=";", encoding="ISO-8859-1")
   return df

def transformar_valores (df_silver):
   df_silver = (
    df_silver

    # Data da venda
    .withColumn(
        "DataVenda",
        F.to_date(F.col("DataVenda"), "dd/MM/yyyy")
    )

    # Preço unitário
    .withColumn(
        "PrecoUnitario",
        F.regexp_replace(
            F.col("PrecoUnitario"), ",", "."
        ).cast("double")
    )

    # Custo unitário
    .withColumn(
        "CustoUnitario",
        F.regexp_replace(
            F.col("CustoUnitario"), ",", "."
        ).cast("double")
    )

    # Quantidade vendida
    .withColumn(
        "Qtd_Vendida",
        F.col("Qtd_Vendida").cast("int")
    )
)
   return df_silver

In [0]:
%python
# ==============================
# IMPORTANDO E CARREGANDO OS DADOS
# EXIBINDO PRIMEIRAS LINHAS
# ==============================

df_spark = spark.createDataFrame(importar_dataset())
display(df_spark.limit(5))

In [0]:
%python
# ==============================
# VERIFICANDO COLUNAS VAZIAS
# ==============================
total_linhas = df_spark.count()

for c in df_spark.columns:
    qtd_vazios = df_spark.filter(
        F.col(c).isNull() |
        (F.trim(F.col(c).cast("string")) == "")
    ).count()

    if qtd_vazios == total_linhas:
        print(f"Coluna totalmente vazia: {c}")

Foi identificada uma coluna sem conteúdo útil (Unnamed: 10), contendo apenas valores ausentes. O atributo será removido durante a transformação da camada Bronze para a camada Silver.


In [0]:
%python
# ==============================
# VERIFICANDO LINHAS VAZIAS
# ==============================

colunas_dados = [c for c in df_spark.columns if c != "Unnamed: 10"]

condicao_vazia = " AND ".join([
    f"(TRIM(CAST({c} AS STRING)) = '' OR {c} IS NULL)"
    for c in colunas_dados
])

df_linhas_vazias = df_spark.filter(F.expr(condicao_vazia))

print("Quantidade de linhas vazias:", df_linhas_vazias.count())

display(df_linhas_vazias)

Durante a análise da camada Bronze foram identificadas linhas completamente vazias ao final do arquivo CSV. Como essas linhas não continham informações de negócio, elas serão removidas durante a transformação para a camada Silver, garantindo maior qualidade e consistência dos dados.

In [0]:
%python
# ==============================
# DEMOSTRANDO O ESQUEMA
# ==============================

df_spark.printSchema()

Identificação e Tratamento de Tipos de Dados

Durante a análise inicial da camada Bronze, foi realizada a inspeção da estrutura dos dados por meio do comando printSchema(). Essa verificação permitiu identificar inconsistências entre os tipos de dados carregados e o significado de negócio de alguns atributos.

Observou-se que as colunas DataVenda, PrecoUnitario e CustoUnitario foram importadas como texto (string), embora representem, respectivamente, uma data e valores monetários. A manutenção desses atributos como texto pode comprometer a execução de análises temporais, cálculos financeiros e agregações estatísticas, além de dificultar a aplicação de regras de qualidade dos dados.

Também foi identificado que a coluna Qtd_Vendida foi carregada como número decimal (double). Entretanto, por representar a quantidade de unidades vendidas em cada transação, esse atributo será convertido para um tipo numérico inteiro mais adequado.

Dessa forma, na camada Silver, serão realizadas as conversões necessárias para adequar cada atributo ao seu domínio de negócio. Essa etapa tem como objetivo garantir maior consistência dos dados, melhorar a qualidade das análises e permitir a criação de métricas derivadas, como faturamento, custo total e lucro, de forma confiável.

In [0]:
%python
# ==============================
# VERIFICANDO ESPAÇOS NOS CAMPOS DE TEXTOS
# ==============================

colunas_texto = [
    "Produto",
    "Categoria",
    "Marca",
    "NomeCliente",
    "Pais",
    "Continente"
]

for c in colunas_texto:
    qtd = (
        df_spark
        .filter(F.col(c) != F.trim(F.col(c)))
        .count()
    )

    print(f"{c}: {qtd} registros com espaços extras")

Padronização de valores textuais

Durante a análise de qualidade dos atributos textuais, foi verificada a presença de espaços excedentes no início ou no final dos valores armazenados. A validação foi realizada nas colunas Produto, Categoria, Marca, NomeCliente, Pais e Continente.

Os resultados demonstraram que apenas o atributo Marca apresentou inconsistências desse tipo, com 14.200 registros contendo espaços extras.

Esse tipo de inconsistência pode afetar agrupamentos, filtros e agregações, fazendo com que valores visualmente iguais sejam tratados como categorias distintas. Por esse motivo, na camada Silver, a coluna Marca será ajustada, garantindo maior padronização e consistência dos dados.

In [0]:
%python
# ==============================
# VERIFICANDO REGISTROS DUPLICADOS
# ==============================

total_registros = df_spark.count()
total_distintos = df_spark.dropDuplicates().count()

qtd_duplicados = total_registros - total_distintos

print("Total de registros:", total_registros)
print("Registros distintos:", total_distintos)
print("Registros duplicados:", qtd_duplicados)

Análise de Registros Duplicados

Durante a avaliação da qualidade dos dados, foi realizada uma análise para identificar possíveis registros duplicados no conjunto de dados. Para isso, comparou-se a quantidade total de registros com a quantidade de registros distintos presentes na base.

A análise identificou 203.888 registros totais, dos quais 171.094 são distintos, resultando em 32.794 registros considerados duplicados.

Entretanto, antes de realizar qualquer remoção, é necessário avaliar a natureza dessas ocorrências. Em bases transacionais de vendas, registros aparentemente idênticos podem representar transações legítimas realizadas em momentos distintos e, portanto, não devem ser removidos automaticamente apenas por apresentarem os mesmos valores em seus atributos.

Dessa forma, os registros duplicados serão analisados durante a etapa de transformação para a camada Silver, a fim de determinar se representam duplicidades técnicas decorrentes do processo de coleta e armazenamento ou se correspondem a eventos de negócio válidos. Somente após essa validação será definida a estratégia de tratamento mais adequada.

In [0]:
%python
# ==============================
# REALIZANDO UMA AVALIAÇÃO MAIS PROFUNDA DA DUPLICIDADE
# ==============================
duplicados = (
    df_spark.groupBy(df_spark.columns)
            .count()
            .filter("count > 1")
            .orderBy("count", ascending=False)
)

A análise de duplicidades identificou 32.794 registros repetidos. A investigação demonstrou a existência de registros idênticos em todos os atributos, alguns deles ocorrendo até 48 vezes. Considerando que o conjunto de dados não possui identificador único de transação e que a repetição integral de todos os atributos caracteriza forte indício de duplicidade técnica, esses registros serão removidos durante a transformação para a camada Silver, mantendo apenas uma ocorrência de cada registro.

### Resumo dos problemas encontrados

| Problema         | Evidência                             | Ação na Silver       |
| ---------------- | ------------------------------------- | -------------------- |
| Coluna vazia     | `Unnamed: 10`                         | Remover              |
| Linhas vazias    | Registros sem conteúdo                | Remover              |
| Tipos incorretos | Datas e valores monetários como texto | Converter            |
| Espaços extras   | 14.200 registros em `Marca`           | Aplicar trim         |
| Duplicidades     | 32.794 registros duplicados           | Remover duplicidades |


### 🎯 Conclusão da Camada Bronze

A análise da camada Bronze permitiu identificar diversos problemas de qualidade presentes nos dados originalmente coletados. Foram encontrados uma coluna sem conteúdo útil, linhas vazias, inconsistências nos tipos de dados, espaços excedentes em atributos textuais e registros duplicados.

Nenhuma alteração foi realizada nesta etapa, uma vez que o objetivo da camada Bronze é preservar os dados em seu estado original. As correções identificadas serão aplicadas durante a transformação para a camada Silver.

### Camada Silver

A camada Silver tem como objetivo transformar os dados provenientes da camada Bronze, realizando processos de limpeza, padronização e adequação dos tipos de dados. Nesta etapa, os problemas de qualidade identificados são tratados, resultando em dados mais consistentes e preparados para as etapas posteriores de modelagem e análise.

Seguiremos com o tratamento seguindo a ordem da tabela abaixo:

✅ Remover coluna vazia  
✅ Remover linhas vazias  
✅ Remover duplicidades  
✅ Padronizar textos  
✅ Corrigir tipos  
✅ Criar Faturamento

### 1️⃣ Remoção de coluna totalmente vazia

In [0]:
%python
# ==============================
# REMOVENDO COLUNA TODA VAZIA
# ==============================

df_silver = df_spark.drop("Unnamed: 10")

display(df_silver.limit(10))

In [0]:
%python
df_silver.printSchema()
print(df_silver.columns)



Durante a análise da camada Bronze foi identificada a coluna Unnamed: 10, sem informação relevante para o negócio e totalmente sem valores.

### 2️⃣ Remoção de linhas completamente vazias
Durante a análise da camada Bronze foram identificados registros completamente vazios, sem qualquer informação relevante para o conjunto de dados. Como esses registros não representam eventos de negócio e não contribuem para as análises, eles serão removidos na camada Silver.

In [0]:
%python
# ==============================
# REMOVENDO LINHAS EM QUE TODOS DADOS ESTEJAM VAZIOS
# ==============================

# Quantidade antes do tratamento
total_antes = df_silver.count()

# Remove registros em que todas as colunas são nulas ou vazias
condicao_linha_vazia = F.expr(
    " AND ".join([
        f"({c} IS NULL OR TRIM(CAST({c} AS STRING)) = '')"
        for c in df_silver.columns
    ])
)

df_silver = df_silver.filter(~condicao_linha_vazia)

# Quantidade após o tratamento
total_depois = df_silver.count()

print(f"Registros antes: {total_antes}")
print(f"Registros depois: {total_depois}")
print(f"Registros removidos: {total_antes - total_depois}")

Após a aplicação do tratamento, foram removidos 6 registros completamente vazios. A camada Silver passou, portanto, a conter 203.882 registros válidos nesta etapa do pipeline.

### 3️⃣ Tratamento dos registros duplicados
Durante a análise da camada Bronze foram identificados 32.794 registros duplicados. A investigação demonstrou a existência de registros idênticos em todos os atributos, inclusive com algumas combinações apresentando diversas ocorrências.

Considerando que a base não possui um identificador único de transação que permita distinguir essas ocorrências e que a repetição integral dos atributos apresenta forte indício de duplicidade técnica, optou-se pela remoção dos registros duplicados durante a transformação para a camada Silver, mantendo apenas uma ocorrência de cada registro.

### OBS:
Na Bronze, a contagem de duplicidades considerava também aquelas linhas completamente vazias e a coluna `Unnamed:` 10. Como já fizemos tratamentos anteriores na Silver, o número removido nesta etapa pode ser ligeiramente diferente.

In [0]:
%python
# ==============================
# REMOVENDO REGISTROS DUPLIUCADOS
# ==============================

# Quantidade antes do tratamento
total_antes = df_silver.count()

# Remove registros duplicados
df_silver = df_silver.dropDuplicates()

# Quantidade após o tratamento
total_depois = df_silver.count()

print(f"Registros antes: {total_antes}")
print(f"Registros depois: {total_depois}")
print(f"Registros duplicados removidos: {total_antes - total_depois}")

In [0]:
%python
# ==============================
# VALIDAÇÃO SE AINDA EXISTEM DADOS DUPLICADOS
# ==============================

duplicados_restantes = (
    df_silver.count()
    - df_silver.dropDuplicates().count()
)

print(f"Duplicidades restantes: {duplicados_restantes}")

### 4️⃣ Padronização dos campos textuais
Durante a análise de qualidade da camada Bronze, foi identificada a presença de espaços excedentes em 14.200 registros do atributo Marca. Embora esses espaços não alterem visualmente o conteúdo, podem fazer com que valores equivalentes sejam interpretados como categorias distintas em operações de agrupamento, filtragem e agregação.

Na camada Silver, esses valores serão padronizados por meio da remoção dos espaços localizados no início e no final das strings, garantindo maior consistência do atributo.

In [0]:
%python
# ==============================
# PADRONIZAR OS DADOS DO ATRIBUTO MARCA: REMOVENDO OS ESPAÇOS
# ==============================

df_silver = df_silver.withColumn(
    "Marca",
    F.trim(F.col("Marca"))
)

In [0]:
%python
# ==============================
# VALIDAÇÃO SE AINDA EXISTEM DADOS COM ESPAÇOS NO ATRIBUTO MARCA
# ==============================

qtd_espacos = (
    df_silver
    .filter(F.col("Marca") != F.trim(F.col("Marca")))
    .count()
)

print(f"Registros com espaços extras em Marca: {qtd_espacos}")

### 5️⃣ Correção dos tipos de dados
Durante a análise da camada Bronze foram identificados atributos cujos tipos de dados não estavam adequados aos respectivos domínios de negócio. A coluna `DataVenda` foi carregada como texto, assim como os atributos monetários `PrecoUnitario` e `CustoUnitario`. Já `Qtd_Vendida`, apesar de representar uma quantidade de unidades, foi carregada como número decimal.

Na camada Silver, esses atributos serão convertidos para tipos mais adequados. A data será transformada para um tipo próprio de data, os valores monetários serão convertidos para valores numéricos e a quantidade vendida será representada como número inteiro. Essas transformações permitem a realização adequada de cálculos, agregações e análises temporais nas etapas posteriores do pipeline.

Esperado conforme a tabela abaixo:
| Atributo        | Tipo atual | Tipo desejado |
| --------------- | ---------- | ------------- |
| `DataVenda`     | string     | date          |
| `PrecoUnitario` | string     | double        |
| `CustoUnitario` | string     | double        |
| `Qtd_Vendida`   | double     | integer       |




In [0]:
%python
# ==============================
# TRANSFORMAR OS DADOS DOS ATRIBUTOS:
# DATA DA VENDA
# PREÇO UNITÁRIO
# CUSTO UNITÁRIO
# QUANTIDADE VENDIDA
# FOI CRIADO A FUNÇÃO transformar_valores() PARA TRANSFORMAR OS DADOS
# ==============================

df_silver = transformar_valores(df_silver)
 

In [0]:
%python
# ==============================
# VALIDAÇÃO DA TRANSFORMAÇÃO DOS DADOS
# ==============================

df_silver.printSchema()

### 6️⃣ Criação do atributo Faturamento
O atributo Faturamento não está presente na base original e será criado na camada Silver como um atributo derivado. Seu cálculo será realizado a partir da multiplicação entre o preço unitário do produto e a quantidade vendida em cada transação.

A criação desse atributo tem como objetivo enriquecer o conjunto de dados e permitir análises financeiras posteriores, como faturamento por produto, categoria, período e região.

In [0]:
%python
# ==============================
# CRIAÇÃO DO ATRIBUTO FATURAMENTO
# ==============================

df_silver = df_silver.withColumn(
    "Faturamento",
    F.col("PrecoUnitario") * F.col("Qtd_Vendida")
)

In [0]:
%python
# ==============================
# VALIDAÇÃO DA CRIAÇÃO DO ATRIBUTO FATURAMENTO
# ==============================

display(
    df_silver.select(
        "Produto",
        "PrecoUnitario",
        "Qtd_Vendida",
        "Faturamento"
    ).limit(10)
)

In [0]:
%python
# ==============================
# VALIDAÇÃO SE EXISTEM DADOS COM FATURAMENTO NULO
# ==============================

qtd_nulos_faturamento = (
    df_silver
    .filter(F.col("Faturamento").isNull())
    .count()
)

print(f"Registros com Faturamento nulo: {qtd_nulos_faturamento}")

### 7️⃣ Criação do atributo CustoTotal
Assim como o faturamento, o custo total de cada venda não está disponível diretamente na base original. Dessa forma, será criado na camada Silver o atributo derivado `CustoTotal`, calculado a partir da multiplicação entre o custo unitário do produto e a quantidade vendida.

Esse atributo permitirá avaliar o custo associado às vendas e servirá posteriormente como base para o cálculo do lucro, além de possibilitar análises de custos por produto, categoria, período e região.

In [0]:
%python
# ==============================
# CRIAÇÃO DO ATRIBUTO CUSTO TOTAL
# =============================

df_silver = df_silver.withColumn(
    "CustoTotal",
    F.round(
        F.col("CustoUnitario") * F.col("Qtd_Vendida"),
        2
    )
)

In [0]:
%python
# ==============================
# VALIDAÇÃO DA CRIAÇÃO DO ATRIBUTO CUSTO TOTAL
# ==============================

display(
    df_silver.select(
        "Produto",
        "CustoUnitario",
        "Qtd_Vendida",
        "CustoTotal"
    ).limit(10)
)

In [0]:
%python
# ==============================
# VALIDAÇÃO SE EXISTEM DADOS NULOS NO ATRIBUTO CUSTO TOTAL
# ==============================

qtd_nulos_custo = (
    df_silver
    .filter(F.col("CustoTotal").isNull())
    .count()
)

print(f"Registros com CustoTotal nulo: {qtd_nulos_custo}")

### 8️⃣ Criação do atributo Lucro
Após a criação dos atributos `Faturamento` e `CustoTotal`, será criado também o atributo derivado `Lucro`. Esse indicador representa o resultado financeiro obtido em cada registro de venda e será calculado pela diferença entre o faturamento gerado e o custo total associado à transação.

A inclusão desse atributo amplia as possibilidades de análise da camada Silver, permitindo avaliar não apenas o volume de vendas e a receita gerada, mas também a rentabilidade por produto, categoria, período e região.

In [0]:
%python
# ==============================
# CRIAÇÃO DO ATRIBUTO LUCRO
# =============================

df_silver = df_silver.withColumn(
    "Lucro",
    F.round(
        F.col("Faturamento") - F.col("CustoTotal"),
        2
    )
)

In [0]:
%python
# ==============================
# VALIDAÇÃO DA CRIAÇÃO DO ATRIBUTO LUCRO
# ==============================

display(
    df_silver.select(
        "Produto",
        "Faturamento",
        "CustoTotal",
        "Lucro"
    ).limit(10)
)

In [0]:
%python
# ==============================
# VALIDAÇÃO SE EXISTEM DADOS NULOS NO ATRIBUTO LUCRO
# ==============================

qtd_nulos_lucro = (
    df_silver
    .filter(F.col("Lucro").isNull())
    .count()
)

print(f"Registros com Lucro nulo: {qtd_nulos_lucro}")

### 🎯 Conclusão da Camada Silver
A camada Silver teve como objetivo transformar os dados brutos provenientes da camada Bronze em um conjunto de dados mais consistente, padronizado e adequado para utilização nas etapas posteriores.

Nesta etapa, foram tratados os problemas de qualidade previamente identificados, incluindo a remoção de coluna e registros sem conteúdo útil, o tratamento de registros duplicados, a padronização de campos textuais e a correção dos tipos de dados conforme seus respectivos domínios de negócio.

Além dos tratamentos de qualidade, o conjunto de dados foi enriquecido com os atributos derivados `Faturamento`, `CustoTotal` e `Lucro`, ampliando as possibilidades de análise financeira e de desempenho das vendas.

Ao final dessas transformações, a camada Silver passa a disponibilizar dados tratados, consistentes e enriquecidos, servindo como fonte confiável para a modelagem e construção da camada Gold, onde os dados serão estruturados para atender às análises e perguntas de negócio definidas no MVP.
